# Create Panel A

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


# ============================================================
# Step 1. File names
# ============================================================


SEEA_INPUT = Path("combined_availability_seea.csv")
WORLDBANK_CLASSIFICATION = Path("worldbank_classification.csv")

COUNTRY_OUTPUT = Path("step4_country_seea_account_coverage.csv")
REGION_OUTPUT = Path("step4_region_seea_account_average_coverage.csv")
INCOME_OUTPUT = Path("step4_income_seea_account_average_coverage.csv")
LONG_OUTPUT = Path("step4_region_income_seea_account_average_coverage_long.csv")

PANEL_B_PNG = Path("step4_panel_B_region_seea_coverage.png")
PANEL_BX_PNG = Path("step4_panel_Bx_income_seea_coverage.png")
COMBINED_PNG = Path("step4_panel_B_and_Bx_seea_coverage.png")
COMBINED_SVG = Path("step4_panel_B_and_Bx_seea_coverage.svg")
COMBINED_PDF = Path("step4_panel_B_and_Bx_seea_coverage.pdf")


# ============================================================
# Step 2. SEEA account settings
# ============================================================

ACCOUNT_COUNT = 36

STATUS_SUFFIXES = {
    "Reported SEEA accounts": "_reported",
    "Easily accessible accounts": "_available",
    "Found but not reported": "_nonreport_avail",
}

STATUS_LABELS = {
    "Reported SEEA accounts": "Reported\nSEEA accounts",
    "Easily accessible accounts": "Easily accessible\naccounts",
    "Found but not reported": "Found but not reported\nin Central Framework\nassessment",
}


# ============================================================
# Step 3. Region and income-group order
# ============================================================

REGION_ORDER = [
    "Eurostat + UK",
    "Non-Eurostat Europe & Central Asia",
    "Middle East, North Africa, Afghanistan & Pakistan",
    "Sub-Saharan Africa",
    "South Asia",
    "East Asia & Pacific",
    "North America",
    "Latin America & Caribbean",
]

INCOME_ORDER = [
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "High income",
    "Unclassified income",
]

REGION_LABELS = {
    "Middle East, North Africa, Afghanistan & Pakistan": "MENA,\nAfghanistan & Pakistan",
    "Latin America & Caribbean": "Latin America &\nCaribbean",
    "Europe & Central Asia": "Europe &\nCentral Asia",
    "East Asia & Pacific": "East Asia &\nPacific",
    "Sub-Saharan Africa": "Sub-Saharan\nAfrica",
}


# ============================================================
# Step 4. Country-name matching rules
# ============================================================

COUNTRY_NAME_MAP = {
    "Czechia": "Czech Republic",
    "Lao People's Democratic Republic": "Lao PDR",
    "Republic of Korea": "South Korea",
    "Republic of Moldova": "Moldova",
    "Russian Federation": "Russia",
    "Slovakia": "Slovak Republic",
    "State of Palestine": "West Bank and Gaza",
    "Türkiye": "Turkey",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Montserrat": "United Kingdom",
    "United Republic of Tanzania": "Tanzania",
    "United States of America": "United States",
    "Congo": "Congo Rep.",
}


# ============================================================
# Step 5. Helper functions
# ============================================================

def clean_binary_frame(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .clip(lower=0, upper=1)
    )


def load_country_coverage() -> pd.DataFrame:
    seea = pd.read_csv(SEEA_INPUT)
    wb = pd.read_csv(WORLDBANK_CLASSIFICATION)

    seea = seea[seea["Country or area"].astype(str).ne("Global")].copy()

    seea["Country or area"] = seea["Country or area"].astype(str).str.strip()
    seea["World Bank country name"] = (
        seea["Country or area"]
        .replace(COUNTRY_NAME_MAP)
        .str.strip()
    )

    wb["Country"] = wb["Country"].astype(str).str.strip()

    reported_cols = [col for col in seea.columns if col.endswith("_reported")]
    available_cols = [col for col in seea.columns if col.endswith("_available")]
    nonreport_cols = [col for col in seea.columns if col.endswith("_nonreport_avail")]

    counts = {
        "_reported": len(reported_cols),
        "_available": len(available_cols),
        "_nonreport_avail": len(nonreport_cols),
    }

    expected_counts = {
        suffix: ACCOUNT_COUNT
        for suffix in STATUS_SUFFIXES.values()
    }

    if counts != expected_counts:
        raise ValueError(
            f"Expected 36 account columns for each SEEA status, found: {counts}"
        )

    coverage = seea[["Country or area", "World Bank country name", "Region"]].copy()
    coverage = coverage.rename(columns={"Region": "SEEA Region"})

    for label, suffix in STATUS_SUFFIXES.items():
        cols = [col for col in seea.columns if col.endswith(suffix)]
        count_col = f"{label}_count"

        coverage[count_col] = clean_binary_frame(seea[cols]).sum(axis=1).astype(int)
        coverage[label] = (coverage[count_col] / ACCOUNT_COUNT * 100).round(2)
        coverage[f"{label}_denominator"] = ACCOUNT_COUNT

    coverage = coverage.merge(
        wb[["Country", "Region", "Income"]],
        left_on="World Bank country name",
        right_on="Country",
        how="left",
    )

    coverage["Region"] = coverage["Region"].fillna(coverage["SEEA Region"])
    coverage["Income"] = coverage["Income"].fillna("Unclassified income")
    coverage = coverage.drop(columns=["Country"])

    leading = [
        "Country or area",
        "World Bank country name",
        "Region",
        "Income",
        "SEEA Region",
    ]

    other = [col for col in coverage.columns if col not in leading]

    return coverage[leading + other]


def average_by_group(country_coverage: pd.DataFrame, group_column: str) -> pd.DataFrame:
    metric_cols = list(STATUS_SUFFIXES.keys())

    grouped = (
        country_coverage.groupby(group_column, dropna=False)[metric_cols]
        .mean()
        .round(2)
        .reset_index()
    )

    grouped.insert(
        1,
        "country_count",
        country_coverage.groupby(group_column, dropna=False)["Country or area"]
        .nunique()
        .values,
    )

    return grouped


def order_group(df: pd.DataFrame, group_column: str, order: list[str]) -> pd.DataFrame:
    ordered = (
        df.set_index(group_column)
        .reindex(order)
        .dropna(how="all", subset=list(STATUS_SUFFIXES.keys()))
        .reset_index()
    )

    return ordered


def build_long_output(region_avg: pd.DataFrame, income_avg: pd.DataFrame) -> pd.DataFrame:
    metric_cols = list(STATUS_SUFFIXES.keys())

    region_long = region_avg.melt(
        id_vars=["Region", "country_count"],
        value_vars=metric_cols,
        var_name="seea_status",
        value_name="average_country_coverage_percent",
    ).rename(columns={"Region": "group"})

    region_long.insert(0, "group_type", "Region")

    income_long = income_avg.melt(
        id_vars=["Income", "country_count"],
        value_vars=metric_cols,
        var_name="seea_status",
        value_name="average_country_coverage_percent",
    ).rename(columns={"Income": "group"})

    income_long.insert(0, "group_type", "Income")

    return pd.concat([region_long, income_long], ignore_index=True)


def annotate_heatmap(ax, values: pd.DataFrame) -> None:
    for y in range(values.shape[0]):
        for x in range(values.shape[1]):
            value = values.iat[y, x]
            text_color = "white" if value >= 55 else "#1f2933"

            ax.text(
                x,
                y,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=9,
                color=text_color,
            )


def draw_panel(
    ax,
    df: pd.DataFrame,
    group_column: str,
    title: str,
    y_label_map: dict[str, str] | None = None,
):
    metric_cols = list(STATUS_SUFFIXES.keys())
    values = df[metric_cols].astype(float)

    image = ax.imshow(
        values,
        vmin=0,
        vmax=100,
        cmap="Blues",
        aspect="auto",
    )

    x_labels = [STATUS_LABELS[col] for col in metric_cols]
    y_labels = [str(value) for value in df[group_column]]

    if y_label_map:
        y_labels = [y_label_map.get(value, value) for value in y_labels]

    y_labels = [
        f"{label}\n(n={count})"
        for label, count in zip(
            y_labels,
            df["country_count"].astype(int).tolist(),
        )
    ]

    ax.set_title(title, loc="left", fontsize=14, fontweight="bold", pad=12)
    ax.set_xticks(range(len(metric_cols)))
    ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel("SEEA account coverage measure", fontsize=10, labelpad=8)
    ax.set_ylabel(group_column, fontsize=10, labelpad=8)

    ax.set_xticks([x - 0.5 for x in range(1, len(metric_cols))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(y_labels))], minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="both", length=0)

    for spine in ax.spines.values():
        spine.set_visible(False)

    annotate_heatmap(ax, values)

    return image


def save_single_panel(
    df: pd.DataFrame,
    group_column: str,
    title: str,
    output_path: Path,
    y_label_map: dict[str, str] | None = None,
) -> None:
    height = max(4.0, 1.0 + len(df) * 0.75)

    fig, ax = plt.subplots(figsize=(9.5, height), constrained_layout=True)

    image = draw_panel(
        ax,
        df,
        group_column,
        title,
        y_label_map,
    )

    cbar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Average country coverage (%)", fontsize=9)

    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Step 6. Run analysis and create figures
# ============================================================

country_coverage = load_country_coverage()

region_avg = order_group(
    average_by_group(country_coverage, "Region"),
    "Region",
    REGION_ORDER,
)

income_avg = order_group(
    average_by_group(country_coverage, "Income"),
    "Income",
    INCOME_ORDER,
)

long_output = build_long_output(region_avg, income_avg)

country_coverage.to_csv(COUNTRY_OUTPUT, index=False, encoding="utf-8-sig")
region_avg.to_csv(REGION_OUTPUT, index=False, encoding="utf-8-sig")
income_avg.to_csv(INCOME_OUTPUT, index=False, encoding="utf-8-sig")
long_output.to_csv(LONG_OUTPUT, index=False, encoding="utf-8-sig")

save_single_panel(
    region_avg,
    "Region",
    "Panel B. Regional country average SEEA account coverage",
    PANEL_B_PNG,
    REGION_LABELS,
)

save_single_panel(
    income_avg,
    "Income",
    "Panel Bx. Income-group country average SEEA account coverage",
    PANEL_BX_PNG,
)

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(10.5, 9.5),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1.25, 1.0]},
)

image = draw_panel(
    axes[0],
    region_avg,
    "Region",
    "B. Regional country average SEEA account coverage",
    REGION_LABELS,
)

draw_panel(
    axes[1],
    income_avg,
    "Income",
    "Bx. Income-group country average SEEA account coverage",
)

cbar = fig.colorbar(image, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Average country coverage (%)", fontsize=10)

fig.savefig(COMBINED_PNG, dpi=300, bbox_inches="tight")
fig.savefig(COMBINED_SVG, bbox_inches="tight")
fig.savefig(COMBINED_PDF, bbox_inches="tight")

plt.close(fig)

unmatched = country_coverage.loc[
    country_coverage["Income"].eq("Unclassified income"),
    ["Country or area", "World Bank country name", "Region", "Income"],
]

print("Step 4 Panel B and Bx SEEA coverage complete")
print(f"Countries used: {len(country_coverage):,}")
print(f"SEEA account denominator: {ACCOUNT_COUNT}")

if not unmatched.empty:
    print("Countries without World Bank income match:")
    print(unmatched.to_string(index=False))

print()
print(f"Country output: {COUNTRY_OUTPUT}")
print(f"Region output: {REGION_OUTPUT}")
print(f"Income output: {INCOME_OUTPUT}")
print(f"Long output: {LONG_OUTPUT}")
print(f"Panel B: {PANEL_B_PNG}")
print(f"Panel Bx: {PANEL_BX_PNG}")
print(f"Combined PNG: {COMBINED_PNG}")
print(f"Combined SVG: {COMBINED_SVG}")
print(f"Combined PDF: {COMBINED_PDF}")

Step 4 Panel B and Bx SEEA coverage complete
Countries used: 120
SEEA account denominator: 36

Country output: step4_country_seea_account_coverage.csv
Region output: step4_region_seea_account_average_coverage.csv
Income output: step4_income_seea_account_average_coverage.csv
Long output: step4_region_income_seea_account_average_coverage_long.csv
Panel B: step4_panel_B_region_seea_coverage.png
Panel Bx: step4_panel_Bx_income_seea_coverage.png
Combined PNG: step4_panel_B_and_Bx_seea_coverage.png
Combined SVG: step4_panel_B_and_Bx_seea_coverage.svg
Combined PDF: step4_panel_B_and_Bx_seea_coverage.pdf
